In [1]:
import json
import torch
import numpy as np
import pandas as pd
import os 
from def_proj_functions import lang_specific_model_names, lang_specific_models, models, definition_models, languages
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
PATH_TO_DEFS = os.getenv('PATH_TO_DEF_EMBS')


def get_definitions(language, model_name):
    path_to_defs = PATH_TO_DEFS
    with open(f'{path_to_defs}/{language}_definitions_by_{model_name}.json', 'r') as f:
        definitions = json.load(f)

    # process the defs - remove any duplicates
    for word in definitions:
        definitions[word] = list(set([definition.strip() for definition in definitions[word]]))
        # add word: to the start of each definition if not already present
        for i, definition in enumerate(definitions[word]):
            if not definition.lower().startswith(word.split('_')[0].lower() + ':'):
                definitions[word][i] = f"{word.split('_')[0]}: {definition.strip()}"
        
    return definitions




/Users/acw747/Projects/definition_projection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt

from def_proj_functions import load_semeval_df, get_wordtransformer_model, get_wordtransformer_embeddings

langs = ['english','german','swedish','latin','spanish','chinese','norwegian_1','norwegian_2']

lang_words = {}
for lang in langs:
    df = load_semeval_df(lang)
    print(f"{lang} has {len(df)} words.")
    words = df['words'].tolist()
    lang_words[lang] = words

# for every word in english, take the first of the split _
lang_words['english'] = [w.split('_')[0] for w in lang_words['english']]


english has 37 words.
german has 48 words.
swedish has 31 words.
latin has 40 words.
spanish has 100 words.
chinese has 40 words.
norwegian_1 has 40 words.
norwegian_2 has 40 words.


In [3]:
from def_proj_functions import get_positions

def embed_definitions(tok, model, definitions):
    """
    definitions: dict of {word: [definition1, definition2, ...]}
    """
    lang_def_embeddings = {}
    for word, def_list in definitions.items():
        word = word.split('_')[0]  # take the first part before underscore
        def_embs_list = []
        for i, definition in enumerate(def_list):
            # find the position of the word in the definition (case insensitive)
            char_pos = get_positions(definition, word)        
            def_lower = definition.lower()  
            # emb = get_XLL_focused_sentence_embedding_from_pos(def_lower, model, char_pos)
            emb = get_wordtransformer_embeddings((tok, model), [definition], [char_pos], batch_size=1, max_length=128)

            def_embs_list.append(emb) 
        lang_def_embeddings[word] = def_embs_list
    return lang_def_embeddings

In [ ]:
# now, embed the definitions (with the character positions of the words within the definitions, usually near the start)

import warnings
warnings.filterwarnings("ignore")

i = 0
for language in languages:
    print(f"[[Processing language]]: {language}")
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"Using device: {device}")


    lang_encoders = models + lang_specific_models.get(language, [])
    for model in lang_encoders:
        print(f"[Using encoder]: {model}")
        tok,encoder = get_wordtransformer_model(model)
        
        if '/' in model:
            encoder_name = lang_specific_model_names[model]
        else:
            encoder_name = model
        for def_model in definition_models:
                # first check if the definition embeddings already exist
                i += 1
                path_to_save = f'{PATH_TO_DEFS}/embeddings2/{language}'
                if os.path.exists(f'{path_to_save}/{def_model}_definitions_embedded_by_{encoder_name}.pt'):
                    # print(f"[{i}]Embeddings for {language} definitions from {def_model} with encoder {encoder_name} already exist. Skipping...")
                    continue

                definitions = get_definitions(language, model_name=def_model)

                lang_def_embeddings = embed_definitions(tok, encoder, definitions)
                # order lang_def_embeddings by lang_words[language]
                lang_def_embeddings = {word: lang_def_embeddings[word] for word in lang_words[language]}


                os.makedirs(path_to_save, exist_ok=True)
                torch.save(lang_def_embeddings, f'{path_to_save}/{def_model}_definitions_embedded_by_{encoder_name}.pt')
                print(f"Saved embeddings to {path_to_save}/{def_model}_definitions_embedded_by_{encoder_name}.pt")
i